In [59]:
from google import genai
import os
from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [60]:
import pandas as pd

df = pd.read_csv("Data/data_mobil.csv")

df['text'] = (
    df['brand'] + " " +
    df['model'] + " " +
    df['specs'] + " " +
    df['description']
)


In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['text'])

def tfidf_search(query, top_k=5):
    q_vec = vectorizer.transform([query])
    sim = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx = sim.argsort()[-top_k:][::-1]
    return df.iloc[idx][['brand','model','price_idr','year']]

In [62]:
from rank_bm25 import BM25Okapi

tokenized = [doc.split() for doc in df['text']]
bm25 = BM25Okapi(tokenized)

def bm25_search(query, top_k=5):
    scores = bm25.get_scores(query.split())
    idx = scores.argsort()[-top_k:][::-1]
    return df.iloc[idx][['brand','model','price_idr','year']]

In [63]:
# query test
query_options = [
    "Mobil yang cocok untuk keluarga",
    "Mobil Listrik",
    "Mobil Autopilot",
    "Mobil hemat bahan bakar",
    "Mobil murah dibawah 250 juta dan keluaran tahun 2023"
]

for i, query in enumerate(query_options):
    print(f"Query {i+1}: {query}")
    print("TF-IDF Search Results:")
    print(tfidf_search(query))
    print("\nBM25 Search Results:")
    print(bm25_search(query))
    print("-" * 50)




Query 1: Mobil yang cocok untuk keluarga
TF-IDF Search Results:
         brand    model  price_idr  year
8       Wuling   Air EV  300000000  2024
1        Honda     Brio  180000000  2023
2   Mitsubishi  Xpander  280000000  2024
19    Daihatsu   Terios  260000000  2023
7       Toyota   Feruza  200000000  2022

BM25 Search Results:
         brand    model  price_idr  year
8       Wuling   Air EV  300000000  2024
1        Honda     Brio  180000000  2023
2   Mitsubishi  Xpander  280000000  2024
7       Toyota   Feruza  200000000  2022
19    Daihatsu   Terios  260000000  2023
--------------------------------------------------
Query 2: Mobil Listrik
TF-IDF Search Results:
       brand    model  price_idr  year
16     Tesla  Model 3  900000000  2024
9    Hyundai  Ioniq 5  750000000  2024
8     Wuling   Air EV  300000000  2024
0     Toyota   Avanza  250000000  2024
5   Daihatsu     Ayla  150000000  2023

BM25 Search Results:
       brand    model  price_idr  year
16     Tesla  Model 3  9000000

In [64]:
#  Semantic Search dengan model selain dari model yang ada pada materi ini, misalkan Anda dapat menggunakan Gemini Embedding. Selain itu cobalah untuk simpan hasil embedding dengan Vector Database Pinecone dan ChromaDB
# embedding dengan Gemini Embedding
def get_embedding(text):
        response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=text
        )
        return response.embeddings[0].values

# apply embedding ke dataframe
df['embedding'] = df['text'].apply(get_embedding)





In [65]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_collection(name="mobil_gemini") if "mobil_gemini" in [col.name for col in chroma_client.list_collections()] else chroma_client.create_collection(name="mobil_gemini")
# add to collection in chromadb
for idx, row in df.iterrows():
    collection.add(
        ids=[str(idx)],
        embeddings=[row['embedding']],
        metadatas=[{"brand": row['brand'], "model": row['model'], "price_idr": row['price_idr'], "year": row['year']}]
    )
def semantic_search(query, top_k=5):
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['metadatas']


In [66]:
import os
from pinecone import Pinecone, ServerlessSpec
load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

dimension = len(df['embedding'][0])  # assuming all embeddings have the same dimension

index_name = "mobil-gemini"
if index_name not in [index.name for index in pc.list_indexes()]:
        pc.create_index(
        name=index_name,
        dimension=dimension,  # match embedding dynamically
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
index = pc.index(index_name)

In [69]:

batch_size = 8

for i in range(0, len(df), batch_size):
    end = min(i + batch_size, len(df))

    upsert_data = []
    for j in range(i, end):
        upsert_data.append(
            (
                str(j),
                list(df['embedding'].iloc[j]),  # ✅ pastikan list
                {
                    "brand": str(df['brand'].iloc[j]),
                    "model": str(df['model'].iloc[j]),
                    "price_idr": int(df['price_idr'].iloc[j]),  # ✅ FIX
                    "year": int(df['year'].iloc[j])             # ✅ FIX
                }
            )
        )

    index.upsert(vectors=upsert_data, namespace="mobil")

    print(f"Upserted batch {i} to {end}")

print(index.describe_index_stats())


def pinecone_search(query, top_k=5):
    query_embedding = get_embedding(query)  # ✅ TANPA [0]

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace="mobil"
    )

    return results['matches']


Upserted batch 0 to 8
Upserted batch 8 to 16
Upserted batch 16 to 20
DescribeIndexStatsResponse(dimension=3072, total_vector_count=20, metric='cosine', namespaces=1)


In [72]:
def format_results(query, chroma_results, pinecone_results):
    def format_price(price):
        return f"Rp{price:,.0f}"

    output = []
    output.append(f"🔎 Query: {query}\n")

    # ---- ChromaDB ----
    output.append("✅ ChromaDB Results")
    chroma_items = chroma_results[0] if chroma_results else []

    for i, item in enumerate(chroma_items, 1):
        brand = item.get("brand", "-")
        model = item.get("model", "-")
        year = item.get("year", "-")
        price = format_price(item.get("price_idr", 0))

        output.append(f"{i}. {brand} {model} ({year}) — {price}")

    # ---- Pinecone ----
    output.append("\n✅ Pinecone Results")

    for i, item in enumerate(pinecone_results, 1):
        meta = item.metadata

        brand = meta.get("brand", "-")
        model = meta.get("model", "-")
        year = meta.get("year", "-")
        price = format_price(meta.get("price_idr", 0))
        score = round(item.score, 3)

        output.append(
            f"{i}. {brand} {model} ({year}) — {price} (Score: {score})"
        )

    return "\n".join(output)

In [74]:
# test all semantic search
for i, query in enumerate(query_options):
    chroma_results = semantic_search(query)
    pinecone_results = pinecone_search(query)

    print(format_results(query, chroma_results, pinecone_results))
    print("-" * 50, "\n")

🔎 Query: Mobil yang cocok untuk keluarga

✅ ChromaDB Results
1. Daihatsu Terios (2023) — Rp260,000,000
2. Nissan Livina (2023) — Rp260,000,000
3. Toyota Avanza (2024) — Rp250,000,000
4. Mitsubishi Xpander (2024) — Rp280,000,000
5. Toyota Rush (2023) — Rp270,000,000

✅ Pinecone Results
1. Daihatsu Terios (2023) — Rp260,000,000 (Score: 0.71)
2. Nissan Livina (2023) — Rp260,000,000 (Score: 0.708)
3. Toyota Avanza (2024) — Rp250,000,000 (Score: 0.707)
4. Mitsubishi Xpander (2024) — Rp280,000,000 (Score: 0.706)
5. Toyota Rush (2023) — Rp270,000,000 (Score: 0.67)
-------------------------------------------------- 

🔎 Query: Mobil Listrik

✅ ChromaDB Results
1. Tesla Model 3 (2024) — Rp900,000,000
2. Wuling Air EV (2024) — Rp300,000,000
3. Hyundai Ioniq 5 (2024) — Rp750,000,000
4. Nissan Livina (2023) — Rp260,000,000
5. Suzuki Ertiga (2023) — Rp240,000,000

✅ Pinecone Results
1. Tesla Model 3 (2024) — Rp900,000,000 (Score: 0.688)
2. Wuling Air EV (2024) — Rp300,000,000 (Score: 0.679)
3. Hyund

In [76]:
#  create streamlit app to test semantic search with chromadb and pinecone

import streamlit as st


st.title("Car Search App")
st.write("Keyword search and semantic search")

query = st.text_input("Enter your search query:")

if st.button("Search"):
    st.write("Keyword search results:")
    st.write("TF-IDF Search Results:")
    st.write(tfidf_search(query))
    st.write("BM25 Search Results:")
    st.write(bm25_search(query))
    st.write("semantic search results:")
    chroma_results = semantic_search(query)
    pinecone_results = pinecone_search(query)

    st.write(format_results(query, chroma_results, pinecone_results))


2026-05-18 22:05:07.211 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.212 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.212 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.213 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.214 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 22:05:07.216 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar